# Fundamentos de NLP: Tokenização, Stemming, BoW e Embeddings (PT-BR)

**Ambiente:** Jupyter Local no VS Code  
**Conexao com:** teoria/AULA1_TEORIA.md — Tópicos 1, 2 e 3

Este notebook é uma demonstração interativa dos conceitos teóricos:
tokenização, stemming, Bag-of-Words, embeddings estáticos e contextuais.

Os exemplos usam o domínio jurídico para ilustrar limitações e avanços de cada técnica.


## 1. Configuração do Ambiente

Instalamos as bibliotecas necessárias para processar texto em português.  
O Ollama (se disponível) será usado para demonstrar embeddings via API REST.


In [ ]:
# Instalação das bibliotecas necessárias
%pip install --quiet gensim nltk scikit-learn pandas numpy sentence-transformers FlagEmbedding

import nltk
import pandas as pd
import numpy as np
import time
import gensim.downloader as api 
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder
from FlagEmbedding import FlagModel

# Downloads de dados específicos do NLTK para Português
nltk.download('punkt')
nltk.download('rslp')
nltk.download('punkt_tab')

# Verificar Ollama (opcional — para demonstracao de embeddings via API)
import requests
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=3)
    modelos = [m['name'] for m in r.json().get('models', [])]
    print(f'Ollama disponivel — modelos: {modelos}')
    OLLAMA_OK = True
except Exception:
    print('Ollama nao detectado — demonstracoes via sentence-transformers apenas')
    OLLAMA_OK = False


## 2. Tokenização

Divisão do texto em tokens (palavras ou sentenças).

In [ ]:
texto = "O suspeito foi detido em flagrante. A Constituição garante o direito à defesa."
print("Sentenças:", sent_tokenize(texto, language='portuguese'))
print("Palavras:", word_tokenize(texto, language='portuguese')[:5])

## 3. Stemming

Redução de palavras ao radical (morfologia).

In [ ]:
stemmer = RSLPStemmer()
for p in ["justiça", "jurídico", "juiz"]:
    print(f"{p} -> {stemmer.stem(p)}")

## 4. Bag-of-Words (BoW)

Representação por contagem. Note como a ordem não importa.

In [ ]:
corpus = ["o juiz absolveu o réu", "o réu absolveu o juiz"]
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)
display(pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out(), index=corpus))

## 5. Embeddings Estáticos (Word2Vec, GloVe)

Vetores densos onde a proximidade geométrica indica similaridade.

In [ ]:
try:
    model_glove = api.load("glove-wiki-gigaword-50")
    print("Similaridade entre 'judge' e 'court':", model_glove.similarity('judge', 'court'))
except: print("Erro no download.")

## 6. Embeddings Contextuais (BERT, Sentence-Transformers e BGE-M3)

### O contexto muda tudo
Em 2018, o BERT representou uma ruptura: ao invés de um vetor fixo, cada palavra ganha representações diferentes dependendo da frase.

---

In [ ]:
print("--- 1. Exemplo com Sentence-Transformers (Português) ---")
model_st = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

frases = [
    "O réu foi condenado pelo magistrado.",
    "O juiz sentenciou o acusado.",
    "O policial efetuou a prisão em flagrante."
]

embeddings_st = model_st.encode(frases)

print(f"Similaridade (Juiz vs Sentença): {cosine_similarity([embeddings_st[0]], [embeddings_st[1]])[0][0]:.4f}")
print(f"Similaridade (Juiz vs Policial): {cosine_similarity([embeddings_st[0]], [embeddings_st[2]])[0][0]:.4f}")

In [ ]:
print("\n--- 2. Exemplo com BGE-M3 (Estado da Arte) ---")
try:
    model_bge = FlagModel('BAAI/bge-m3', use_fp16=True)
    embeddings_bge = model_bge.encode(frases)
    
    score_1 = np.dot(embeddings_bge[0], embeddings_bge[1])
    score_2 = np.dot(embeddings_bge[0], embeddings_bge[2])
    
    print(f"Similaridade BGE-M3 (Juiz vs Sentença): {score_1:.4f}")
    print(f"Similaridade BGE-M3 (Juiz vs Policial): {score_2:.4f}")
except Exception as e: 
    print(f"Nota: BGE-M3 pode exigir GPU ou mais memória. Erro: {e}")

### 6.3 Desempenho: Por que Bi-Encoders são o padrão para RAG?

**Pergunta:** Por que usar BERT diretamente (Cross-Encoder) para busca em 100.000 documentos é inviável?

**Resposta:** O BERT original foi projetado como um **Cross-Encoder**: ele precisa de dois textos juntos para produzir um score. Se você tem 100.000 documentos, precisaria rodar o modelo 100.000 vezes para cada busca.

O **Sentence-Transformers (Bi-Encoder)** resolve isso: você faz o embedding dos 100.000 documentos **uma única vez** e armazena. Na busca, faz apenas **1 embedding** da pergunta e compara vetores em milissegundos.

In [26]:
from sentence_transformers import CrossEncoder
import time

print("--- Comparação de Velocidade: Cross-Encoder vs Bi-Encoder ---")
query = "Qual a pena para o crime de peculato?"
docs = [f"Documento jurídico número {i} contendo informações sobre leis penais..." for i in range(100)]

# 1. Simulação Cross-Encoder (Lento)
model_ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
start_ce = time.time()
# Cross-encoder precisa processar cada par (query, doc)
scores_ce = model_ce.predict([(query, doc) for doc in docs])
end_ce = time.time()
time_ce = end_ce - start_ce
print(f"Tempo Cross-Encoder (100 documentos): {time_ce:.4f} segundos")

# 2. Simulação Bi-Encoder (Rápido)
# Nota: Em um sistema real, os docs já estariam encodados.
doc_embeddings = model_st.encode(docs)

start_bi = time.time()
query_embedding = model_st.encode(query)
similarities = cosine_similarity([query_embedding], doc_embeddings)
end_bi = time.time()
time_bi = end_bi - start_bi
print(f"Tempo Bi-Encoder (100 documentos): {time_bi:.4f} segundos")

print(f"\nO Bi-Encoder foi aproximadamente {time_ce/time_bi:.1f}x mais rápido para apenas 100 docs.")
print("Imagine a diferença para 100.000 documentos!")

--- Comparação de Velocidade: Cross-Encoder vs Bi-Encoder ---


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tempo Cross-Encoder (100 documentos): 1.4282 segundos
Tempo Bi-Encoder (100 documentos): 0.0432 segundos

O Bi-Encoder foi aproximadamente 33.1x mais rápido para apenas 100 docs.
Imagine a diferença para 100.000 documentos!


## 7. Por que usar em RAG?

A evolução de BoW -> Word2Vec -> BGE-M3 permitiu que sistemas de IA jurídica não apenas "contem palavras", mas entendam conceitos complexos, sinônimos e contextos específicos, tornando a recuperação de documentos em RAG muito mais precisa.